# Chạy full pipeline với Chandra2 trên Colab GPU

Notebook này chạy Chandra2 thông qua chính code của repo (`src.pipeline`), không gọi model trực tiếp bên ngoài ingestion contract.

Trước khi **Run All**:

1. Trong VS Code, mở notebook và chọn kernel **Colab** đã gắn GPU L4/A100.
2. Source local đã được đóng gói vào Drive. Notebook sẽ tự giải nén source bundle vào Colab disk, nên không cần GitHub token.
3. Input lấy trực tiếp từ raw artifact của run `a6197afb24dcc5f6` trên Drive.
4. Kết quả từng stage và một file ZIP sẽ được lưu vào Google Drive. Không cần upload/download thủ công trong notebook.

Notebook chạy đúng PDF UET CONNECT 2022. Lượt đầu dùng `ocr_layout`; mỗi block Table có bbox sẽ được crop và OCR lần hai bằng prompt chuyên cho merged cells. Embedding dùng `local_hash`, nên không cần OpenRouter API key.

In [ ]:
from pathlib import Path

# Repo/runtime: drive_zip lấy đúng cả code local chưa commit; git là fallback.
CODE_SOURCE = "drive_zip"  # "drive_zip" hoặc "git"
REPO_URL = "https://github.com/iSE-UET-VNU/AXIOM_DE-RD.git"
BRANCH = "feature/ingestion"
REPO_DIR = Path("/content/AXIOM_DE-RD")
DRIVE_REPO_ZIP = Path(
    "/content/drive/MyDrive/AXIOM_DE-RD/source-bundles/"
    "AXIOM_DE-RD-feature-ingestion-20260723T200133-table-refinement-fix.zip"
)
DRIVE_REPO_FILE_ID = "11ToQOwN0XqxtxpLQBphufvMa66U1ob52"
DRIVE_REPO_SHA256 = (
    "ae080b3eb5af630a8a5e4493ee5693bc1fc73dcc6f56ad43eef6d9c3026db44d"
)

# Input đã có sẵn trên mounted Google Drive
EXPERIMENT_ID = "parser-smoke-uet-connect-2022-table-refinement-20260724"
DRIVE_INPUT = Path(
    "/content/drive/MyDrive/AXIOM_DE-RD/chandra2-pipeline-runs/"
    "a6197afb24dcc5f6/raw/95168ff83bfc14d8"
)
DRIVE_INPUT_SHA256 = None  # Không dùng khi DRIVE_INPUT là thư mục

# Output bền vững trên Drive
DRIVE_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/AXIOM_DE-RD/chandra2-pipeline-runs"
)

# Chandra2 HF local inference
MODEL_CHECKPOINT = "datalab-to/chandra-ocr-2"
MAX_FILES = 1          # Thư mục input chỉ chứa PDF UET CONNECT 2022
MAX_OUTPUT_TOKENS = 12384
REFINE_TABLES = True
TABLE_MAX_OUTPUT_TOKENS = 4096
INCLUDE_IMAGES = True
INCLUDE_HEADERS_FOOTERS = False


In [ ]:
import os
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "Runtime chưa có GPU. Trong VS Code hãy chọn lại Colab kernel có GPU."
    )

gpu_status = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(f"Python: {sys.version.split()[0]}")
print(f"GPU: {gpu_status.stdout.strip()}")

from google.colab import drive

drive.mount("/content/drive", force_remount=True)
print(f"Drive input: {DRIVE_INPUT}")


In [ ]:
import base64
import getpass
import hashlib
import io
import json
import shutil
import zipfile
from datetime import datetime, timezone


def run_command(command: list[str], *, cwd: Path | None = None) -> None:
    safe_command = [
        "http.extraHeader=<redacted>"
        if item.startswith("http.extraHeader=")
        else item
        for item in command
    ]
    print("+", " ".join(safe_command))
    result = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True,
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode:
        error = result.stderr.strip() or "Command failed without stderr."
        raise RuntimeError(f"Command failed (exit {result.returncode}):\n{error}")


def move_existing_repo_aside() -> None:
    if not REPO_DIR.exists():
        return
    if REPO_DIR.resolve().parent != Path("/content"):
        raise RuntimeError(f"Refusing to replace unexpected path: {REPO_DIR}")
    suffix = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    backup = REPO_DIR.with_name(f"{REPO_DIR.name}.previous-{suffix}")
    REPO_DIR.rename(backup)
    print(f"Moved previous runtime source to: {backup}")


def download_drive_file(file_id: str, destination: Path) -> None:
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    print("Mounted path chưa thấy bundle; authenticating Drive API by file ID...")
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    service = build("drive", "v3", credentials=credentials, cache_discovery=False)
    metadata = service.files().get(
        fileId=file_id,
        fields="id,name,size",
    ).execute()
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, "wb") as output:
        downloader = MediaIoBaseDownload(output, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"Download: {int(status.progress() * 100)}%")
    print(f"Downloaded {metadata.get('name')} ({metadata.get('size')} bytes)")


if CODE_SOURCE == "drive_zip":
    repo_zip_path = DRIVE_REPO_ZIP
    if not repo_zip_path.is_file():
        repo_zip_path = Path("/content") / DRIVE_REPO_ZIP.name
        try:
            download_drive_file(DRIVE_REPO_FILE_ID, repo_zip_path)
        except Exception as exc:
            raise FileNotFoundError(
                f"Không đọc được source bundle bằng mounted path hoặc file ID. "
                f"Hãy xác nhận Drive API dùng đúng account. Root cause: {exc}"
            ) from exc
    actual_sha256 = hashlib.sha256(repo_zip_path.read_bytes()).hexdigest()
    if actual_sha256 != DRIVE_REPO_SHA256:
        raise RuntimeError(
            f"Source bundle checksum mismatch: {actual_sha256}"
        )
    move_existing_repo_aside()
    REPO_DIR.mkdir(parents=True, exist_ok=False)
    # Tự giải nén với path POSIX: ZIP tạo trên Windows đôi khi dùng \ thay vì /.
    with zipfile.ZipFile(repo_zip_path) as archive:
        safe_root = REPO_DIR.resolve()
        for member in archive.infolist():
            name = member.filename.replace("\\", "/")
            if name.startswith("/"):
                raise RuntimeError(f"ZIP member không an toàn: {member.filename}")
            parts = [part for part in name.split("/") if part not in {"", "."}]
            if not parts:
                continue
            if ".." in parts or ":" in parts[0]:
                raise RuntimeError(f"ZIP member không an toàn: {member.filename}")
            target = REPO_DIR.joinpath(*parts).resolve()
            if safe_root not in target.parents:
                raise RuntimeError(f"ZIP member không an toàn: {member.filename}")
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)
    required_files = ["pyproject.toml", "src/pipeline.py"]
    missing_files = [name for name in required_files if not (REPO_DIR / name).is_file()]
    if missing_files:
        raise RuntimeError(f"Source bundle thiếu file bắt buộc: {missing_files}")
    manifest_path = REPO_DIR / ".colab-source-bundle.json"
    source_manifest = (
        json.loads(manifest_path.read_text(encoding="utf-8-sig"))
        if manifest_path.is_file()
        else {}
    )
    commit = str(source_manifest.get("base_commit") or "local-worktree")
    source_ref = f"drive_zip:{repo_zip_path.name}"
elif CODE_SOURCE == "git":
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
    if not GITHUB_TOKEN:
        try:
            from google.colab import userdata
            GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
        except Exception:
            GITHUB_TOKEN = None
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass.getpass(
            "GitHub token cho private repo (input được ẩn): "
        ).strip()

    def git_command(*arguments: str) -> list[str]:
        if not GITHUB_TOKEN:
            return ["git", *arguments]
        credentials = base64.b64encode(
            f"x-access-token:{GITHUB_TOKEN}".encode("utf-8")
        ).decode("ascii")
        return [
            "git",
            "-c",
            f"http.extraHeader=Authorization: Basic {credentials}",
            *arguments,
        ]

    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        move_existing_repo_aside()
    if not REPO_DIR.exists():
        run_command(
            [
                *git_command("clone"),
                "--depth",
                "1",
                "--branch",
                BRANCH,
                REPO_URL,
                str(REPO_DIR),
            ]
        )
    else:
        run_command(git_command("fetch", "origin", BRANCH), cwd=REPO_DIR)
        run_command(git_command("checkout", BRANCH), cwd=REPO_DIR)
        run_command(
            git_command("pull", "--ff-only", "origin", BRANCH),
            cwd=REPO_DIR,
        )
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    source_ref = f"git:{BRANCH}@{commit}"
else:
    raise ValueError("CODE_SOURCE phải là 'drive_zip' hoặc 'git'.")

print(f"Using source: {source_ref}")


In [ ]:
import importlib.metadata

os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["MODEL_CHECKPOINT"] = MODEL_CHECKPOINT
os.environ["TORCH_DEVICE"] = "cuda"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.pop("TORCH_ATTN", None)

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        ".[chandra2-local]",
    ],
    cwd=REPO_DIR,
)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch không nhận CUDA sau khi cài dependencies.")

gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / (1024 ** 3)
if not torch.cuda.is_bf16_supported():
    raise RuntimeError(f"GPU {gpu.name} không hỗ trợ BF16.")
if gpu_memory_gib < 18:
    raise RuntimeError(
        f"GPU {gpu.name} chỉ có {gpu_memory_gib:.1f} GiB VRAM; "
        "Chandra2 cần tối thiểu khoảng 18 GiB."
    )

print(f"CUDA ready: {gpu.name}, {gpu_memory_gib:.1f} GiB")
print(f"torch={torch.__version__}")
print(f"chandra-ocr={importlib.metadata.version('chandra-ocr')}")


In [ ]:
import zipfile

SUPPORTED_EXTENSIONS = {
    ".pdf", ".png", ".jpg", ".jpeg", ".gif", ".webp", ".tiff"
}
EXTRACT_ROOT = Path("/content/axiom-chandra2-input") / EXPERIMENT_ID

if not DRIVE_INPUT.exists():
    repo_fallback = REPO_DIR / "data" / "raw" / "test"
    if repo_fallback.is_dir():
        INPUT_DIR = repo_fallback
        print(f"Dùng corpus có sẵn trong repo: {INPUT_DIR}")
    else:
        raise FileNotFoundError(
            f"Không tìm thấy {DRIVE_INPUT}. Hãy upload corpus.zip vào Drive "
            "hoặc sửa biến DRIVE_INPUT."
        )
elif DRIVE_INPUT.is_dir():
    INPUT_DIR = DRIVE_INPUT
else:
    if DRIVE_INPUT.suffix.lower() != ".zip":
        raise ValueError("DRIVE_INPUT phải là một thư mục hoặc file .zip.")
    actual_input_sha256 = hashlib.sha256(DRIVE_INPUT.read_bytes()).hexdigest()
    if actual_input_sha256 != DRIVE_INPUT_SHA256:
        raise RuntimeError(
            "SHA-256 của input ZIP không khớp: "
            f"expected={DRIVE_INPUT_SHA256}, actual={actual_input_sha256}"
        )
    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_INPUT) as archive:
        safe_root = EXTRACT_ROOT.resolve()
        for member in archive.infolist():
            target = (EXTRACT_ROOT / member.filename).resolve()
            if target != safe_root and safe_root not in target.parents:
                raise RuntimeError(f"ZIP member không an toàn: {member.filename}")
        archive.extractall(EXTRACT_ROOT)
    corpus_dir = EXTRACT_ROOT / "corpus"
    INPUT_DIR = corpus_dir if corpus_dir.is_dir() else EXTRACT_ROOT

input_files = sorted(
    path
    for path in INPUT_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)
if not input_files:
    raise RuntimeError(f"Không có PDF/image được Chandra2 hỗ trợ trong {INPUT_DIR}.")

print(f"Input directory: {INPUT_DIR}")
print(f"Found {len(input_files)} supported file(s):")
for path in input_files:
    print(f"- {path.relative_to(INPUT_DIR)} ({path.stat().st_size:,} bytes)")


In [ ]:
import copy
import json
from datetime import datetime, timezone

sys.path.insert(0, str(REPO_DIR))
from src.utils.config import load_config

session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOCAL_RUN_ROOT = Path("/content/axiom-chandra2-pipeline") / session_id
CONFIG_PATH = LOCAL_RUN_ROOT / "pipeline.chandra2-colab.json"
LOCAL_RUN_ROOT.mkdir(parents=True, exist_ok=False)

base_config_path = REPO_DIR / "configs" / "pipeline.yaml"
if base_config_path.is_file():
    config = copy.deepcopy(load_config(base_config_path))
    pipeline_config_source = str(base_config_path)
else:
    # Chạy được ngay cả khi source bundle chỉ chứa Python package, không có configs/.
    pipeline_config_source = "embedded Colab fallback"
    print(f"WARNING: missing {base_config_path}; using {pipeline_config_source}.")
    config = {
        "raw_dir": "data/raw",
        "ingested_dir": "data/ingested",
        "cleaned_dir": "data/cleaned",
        "enriched_dir": "data/enriched",
        "embedded_dir": "data/embedded",
        "output_dir": "data/output",
        "parsing": {},
        "logging": {"level": "INFO"},
    }
config["input"] = {"mode": "local_raw"}
config["local_input"] = {
    "path": str(INPUT_DIR),
    "recursive": True,
    "include_hidden": False,
    "include_extensions": sorted(SUPPORTED_EXTENSIONS),
    "max_files": MAX_FILES,
}
for stage in ("raw", "ingested", "cleaned", "enriched", "embedded", "output"):
    config[f"{stage}_dir"] = str(LOCAL_RUN_ROOT / stage)

config["enabled_modules"] = [
    "ingestion",
    "cleaning",
    "enrichment",
    "indexing_cataloging",
    "integration",
    "artifacts",
]
config["parsing"]["provider"] = "chandra2"
config["parsing"]["chandra2"] = {
    "method": "hf",
    "batch_size": 2,
    "max_workers": 1,
    "max_output_tokens": MAX_OUTPUT_TOKENS,
    "max_retries": 0,
    "include_images": INCLUDE_IMAGES,
    "include_headers_footers": INCLUDE_HEADERS_FOOTERS,
    "save_raw_outputs": True,
    "refine_tables": REFINE_TABLES,
    "table_max_output_tokens": TABLE_MAX_OUTPUT_TOKENS,
    "table_crop_margin_ratio": 0.02,
    "table_crop_min_short_side": 1536,
    "table_crop_max_long_side": 3072,
    "table_crop_max_pixels": 6291456,
}

# Smoke test không cần API key; production có thể đổi lại OpenRouter sau.
config["indexing"] = {
    "embeddings": {
        "enabled": True,
        "target_index_types": ["text_chunk", "table", "image"],
        "provider": "local_hash",
        "model": "local-hash-embedding-v1",
        "dimension": 256,
        "batch_size": 32,
        "fail_on_error": True,
    }
}
config["chunking_embedding"] = {
    "chunker": "recursive",
    "chunker_params": {"target": 400},
    "max_rows_per_chunk": 20,
    "embedder": "local_hash",
    "embedder_params": {
        "model": "local-hash-embedding-v1",
        "dimension": 256,
    },
    "retrieval_profile": "hybrid_default",
}

CONFIG_PATH.write_text(
    json.dumps(config, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Runtime config: {CONFIG_PATH}")
print("Provider: chandra2 / method: hf / batch_size: 1")


In [ ]:
import time

os.chdir(REPO_DIR)
from src.pipeline import run_pipeline

started = time.monotonic()
state = run_pipeline(CONFIG_PATH, local_raw=INPUT_DIR)
elapsed_seconds = round(time.monotonic() - started, 3)

run_summary = {
    "run_id": state.run_id,
    "code_source": source_ref,
    "branch": BRANCH,
    "commit": commit,
    "provider": "chandra2",
    "method": "hf",
    "model": MODEL_CHECKPOINT,
    "input_dir": str(INPUT_DIR),
    "input_file_count": (
        len(input_files) if MAX_FILES is None else min(MAX_FILES, len(input_files))
    ),
    "succeeded_documents": len(state.data_objects),
    "quarantined_documents": len(state.quarantined_documents),
    "errors": state.errors,
    "completed_modules": state.completed_modules,
    "elapsed_seconds": elapsed_seconds,
}
print(json.dumps(run_summary, ensure_ascii=False, indent=2, default=str))


In [ ]:
import shutil
import zipfile

DRIVE_RUN_DIR = DRIVE_OUTPUT_ROOT / state.run_id
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)

def resolve_state_path(value: str) -> Path:
    path = Path(value)
    return path if path.is_absolute() else REPO_DIR / path


stage_sources = {
    "raw": resolve_state_path(state.raw_dir),
    "ingested": resolve_state_path(state.ingested_dir),
    "cleaned": resolve_state_path(state.cleaned_dir),
    "enriched": resolve_state_path(state.enriched_dir),
    "embedded": resolve_state_path(state.embedded_dir),
    "output": resolve_state_path(state.output_dir),
}

for stage, source in stage_sources.items():
    if source.exists():
        shutil.copytree(source, DRIVE_RUN_DIR / stage, dirs_exist_ok=True)
        print(f"Saved {stage}: {DRIVE_RUN_DIR / stage}")

summary_path = DRIVE_RUN_DIR / "run_summary.json"
summary_path.write_text(
    json.dumps(run_summary, ensure_ascii=False, indent=2, default=str) + "\n",
    encoding="utf-8",
)
shutil.copy2(CONFIG_PATH, DRIVE_RUN_DIR / CONFIG_PATH.name)

local_zip = Path("/content") / f"{state.run_id}.zip"
with zipfile.ZipFile(local_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for stage, source in stage_sources.items():
        if not source.exists():
            continue
        for path in source.rglob("*"):
            if path.is_file():
                archive.write(path, Path(stage) / path.relative_to(source))
    archive.write(CONFIG_PATH, CONFIG_PATH.name)
    archive.write(summary_path, "run_summary.json")

drive_zip = DRIVE_OUTPUT_ROOT / f"{state.run_id}.zip"
shutil.copy2(local_zip, drive_zip)

print(f"\nDrive run: {DRIVE_RUN_DIR}")
print(f"Drive ZIP: {drive_zip} ({drive_zip.stat().st_size:,} bytes)")
if state.errors:
    print("WARNING: Pipeline có lỗi/quarantine; xem run_summary.json và stage ingested.")


In [ ]:
# Kiểm tra nhanh các output JSON cuối cùng.
output_jsons = sorted((DRIVE_RUN_DIR / "output").rglob("*.json"))
print(f"Final output JSON files: {len(output_jsons)}")
for path in output_jsons[:20]:
    print("-", path.relative_to(DRIVE_RUN_DIR))
if len(output_jsons) > 20:
    print(f"... and {len(output_jsons) - 20} more")

subprocess.run(["nvidia-smi"], check=True)
